In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
import io
import warnings
warnings.filterwarnings('ignore')

# Upload the file
print("Please upload the Excel file (ai_model (4).xlsx):")
uploaded = files.upload()

# Read the Excel file
file_name = list(uploaded.keys())[0]
df = pd.read_excel(io.BytesIO(uploaded[file_name]), sheet_name='Sheet1')

# Clean the data
df.replace('Null', np.nan, inplace=True)
df.replace('null', np.nan, inplace=True)
df.replace('Null', np.nan, inplace=True)

# Drop empty rows
df = df.dropna(how='all')

# Separate Dengue and Chikungunya patients
df_dengue = df[df['Disease Type'].str.contains('Dengue', case=False, na=False)].copy()
df_chik = df[df['Disease Type'].str.contains('Chikungunya', case=False, na=False)].copy()

total_dengue = len(df_dengue)
total_chik = len(df_chik)
total_all = len(df)

print(f"\nTotal Patients: {total_all}")
print(f"Dengue Patients: {total_dengue}")
print(f"Chikungunya Patients: {total_chik}")

Please upload the Excel file (ai_model (4).xlsx):


In [ ]:
# =============================================================================
# 1. GENDER ANALYSIS - Male/Female Ratio
# =============================================================================

print("="*70)
print("1. GENDER RATIO ANALYSIS")
print("="*70)

gender_dengue = df_dengue['Gender'].value_counts()
gender_chik = df_chik['Gender'].value_counts()

male_dengue = gender_dengue.get('Male', 0)
female_dengue = gender_dengue.get('Female', 0)
male_chik = gender_chik.get('Male', 0)
female_chik = gender_chik.get('Female', 0)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Dengue Gender Pie
axes[0].pie([male_dengue, female_dengue], labels=['Male', 'Female'], autopct='%1.1f%%',
           colors=['#FF6B6B', '#4ECDC4'], startangle=90, explode=[0.05, 0])
axes[0].set_title(f'Dengue Gender (n={total_dengue})', fontweight='bold')

# Chikungunya Gender Pie
axes[1].pie([male_chik, female_chik], labels=['Male', 'Female'], autopct='%1.1f%%',
           colors=['#FF6B6B', '#4ECDC4'], startangle=90, explode=[0.05, 0])
axes[1].set_title(f'Chikungunya Gender (n={total_chik})', fontweight='bold')

# Comparison Bar
x = ['Male', 'Female']
width = 0.35
axes[2].bar(np.arange(len(x)) - width/2, [male_dengue, female_dengue], width, label='Dengue', color='#FF6B6B')
axes[2].bar(np.arange(len(x)) + width/2, [male_chik, female_chik], width, label='Chikungunya', color='#4ECDC4')
axes[2].set_xticks(np.arange(len(x)))
axes[2].set_xticklabels(x)
axes[2].set_title('Gender Comparison')
axes[2].legend()

plt.tight_layout()
plt.show()

print(f"\nDengue: Male={male_dengue} ({(male_dengue/total_dengue*100):.1f}%), Female={female_dengue} ({(female_dengue/total_dengue*100):.1f}%), M:F = {male_dengue}:{female_dengue}")
print(f"Chikungunya: Male={male_chik} ({(male_chik/total_chik*100):.1f}%), Female={female_chik} ({(female_chik/total_chik*100):.1f}%), M:F = {male_chik}:{female_chik}")

In [ ]:
# =============================================================================
# 2. SYMPTOMS BY DISEASE TYPE
# =============================================================================

print("\n" + "="*70)
print("2. SYMPTOMS ANALYSIS BY DISEASE TYPE")
print("="*70)

symptom_columns = ['High Fever', 'Headache', 'Muscle Pain', 'Joint Pain', 'Rash', 'Bleeding Menifestations']

symptom_data = []
for symptom in symptom_columns:
    dengue_yes = len(df_dengue[df_dengue[symptom].astype(str).str.lower().str.contains('yes', na=False)])
    chik_yes = len(df_chik[df_chik[symptom].astype(str).str.lower().str.contains('yes', na=False)])
    symptom_data.append({
        'Symptom': symptom,
        'Dengue_Count': dengue_yes,
        'Dengue_Pct': (dengue_yes/total_dengue)*100,
        'Chik_Count': chik_yes,
        'Chik_Pct': (chik_yes/total_chik)*100
    })

symptom_df = pd.DataFrame(symptom_data)

fig, axes = plt.subplots(1, 3, figsize=(16, 6))

# Dengue Symptoms
axes[0].barh(symptom_df['Symptom'], symptom_df['Dengue_Pct'], color='#FF6B6B')
axes[0].set_title(f'Dengue Symptoms (n={total_dengue})', fontweight='bold')
axes[0].set_xlabel('Percentage (%)')
for i, (pct, cnt) in enumerate(zip(symptom_df['Dengue_Pct'], symptom_df['Dengue_Count'])):
    axes[0].text(pct+0.5, i, f'{pct:.1f}% (n={cnt})', va='center')

# Chikungunya Symptoms
axes[1].barh(symptom_df['Symptom'], symptom_df['Chik_Pct'], color='#4ECDC4')
axes[1].set_title(f'Chikungunya Symptoms (n={total_chik})', fontweight='bold')
axes[1].set_xlabel('Percentage (%)')
for i, (pct, cnt) in enumerate(zip(symptom_df['Chik_Pct'], symptom_df['Chik_Count'])):
    axes[1].text(pct+0.5, i, f'{pct:.1f}% (n={cnt})', va='center')

# Comparison
x = np.arange(len(symptom_df))
width = 0.35
axes[2].bar(x - width/2, symptom_df['Dengue_Pct'], width, label='Dengue', color='#FF6B6B')
axes[2].bar(x + width/2, symptom_df['Chik_Pct'], width, label='Chikungunya', color='#4ECDC4')
axes[2].set_xticks(x)
axes[2].set_xticklabels(symptom_df['Symptom'], rotation=45, ha='right')
axes[2].set_ylabel('Percentage (%)')
axes[2].legend()
axes[2].set_title('Symptom Comparison', fontweight='bold')

plt.tight_layout()
plt.show()

print("\nSymptom Summary:")
print(symptom_df[['Symptom', 'Dengue_Pct', 'Chik_Pct']].to_string(index=False))

In [ ]:
# =============================================================================
# 3. TEMPERATURE ANALYSIS
# =============================================================================

print("\n" + "="*70)
print("3. TEMPERATURE - HOT vs COLD RATIO")
print("="*70)

temp_dengue = df_dengue['Temperature'].value_counts()
temp_chik = df_chik['Temperature'].value_counts()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Dengue
colors_temp = ['#FF6B6B', '#4ECDC4', '#FFE66D']
axes[0].pie(temp_dengue.values, labels=temp_dengue.index, autopct='%1.1f%%',
           colors=colors_temp[:len(temp_dengue)], startangle=90, explode=[0.05]*len(temp_dengue))
axes[0].set_title(f'Dengue Temperature (n={total_dengue})', fontweight='bold')

# Chikungunya
axes[1].pie(temp_chik.values, labels=temp_chik.index, autopct='%1.1f%%',
           colors=colors_temp[:len(temp_chik)], startangle=90, explode=[0.05]*len(temp_chik))
axes[1].set_title(f'Chikungunya Temperature (n={total_chik})', fontweight='bold')

# Comparison
all_temps = sorted(set(list(temp_dengue.index) + list(temp_chik.index)))
dengue_temp_vals = [temp_dengue.get(t, 0) for t in all_temps]
chik_temp_vals = [temp_chik.get(t, 0) for t in all_temps]
x = np.arange(len(all_temps))
width = 0.35
axes[2].bar(x - width/2, dengue_temp_vals, width, label='Dengue', color='#FF6B6B')
axes[2].bar(x + width/2, chik_temp_vals, width, label='Chikungunya', color='#4ECDC4')
axes[2].set_xticks(x)
axes[2].set_xticklabels(all_temps)
axes[2].legend()
axes[2].set_title('Temperature Comparison', fontweight='bold')

plt.tight_layout()
plt.show()

hot_dengue = temp_dengue.get('Hot', 0)
cold_dengue = temp_dengue.get('Cold', 0)
hot_chik = temp_chik.get('Hot', 0)
cold_chik = temp_chik.get('Cold', 0)

print(f"\nDengue: Hot={hot_dengue} ({(hot_dengue/total_dengue*100):.1f}%), Cold={cold_dengue} ({(cold_dengue/total_dengue*100):.1f}%)")
print(f"Chikungunya: Hot={hot_chik} ({(hot_chik/total_chik*100):.1f}%), Cold={cold_chik} ({(cold_chik/total_chik*100):.1f}%)")

In [ ]:
# =============================================================================
# 4 & 5. RAINFALL & HOUSING (Combined for efficiency)
# =============================================================================

print("\n" + "="*70)
print("4. RAINFALL FREQUENCY ANALYSIS")
print("="*70)

rainfall_col = None
for col in df.columns:
    if 'Rainfall' in col:
        rainfall_col = col
        break

rain_dengue = df_dengue[rainfall_col].value_counts()
rain_chik = df_chik[rainfall_col].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].pie(rain_dengue.values, labels=rain_dengue.index, autopct='%1.1f%%',
           colors=plt.cm.Blues(np.linspace(0.3, 0.9, len(rain_dengue))), startangle=90)
axes[0].set_title(f'Dengue Rainfall (n={total_dengue})', fontweight='bold')
axes[1].pie(rain_chik.values, labels=rain_chik.index, autopct='%1.1f%%',
           colors=plt.cm.Blues(np.linspace(0.3, 0.9, len(rain_chik))), startangle=90)
axes[1].set_title(f'Chikungunya Rainfall (n={total_chik})', fontweight='bold')
plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("5. HOUSING TYPES / BUILDING MATERIALS")
print("="*70)

housing_dengue = df_dengue['Housing types/ Building materials'].value_counts()
housing_chik = df_chik['Housing types/ Building materials'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].pie(housing_dengue.values, labels=housing_dengue.index, autopct='%1.1f%%',
           colors=plt.cm.Set2(np.linspace(0, 1, len(housing_dengue))), startangle=90)
axes[0].set_title(f'Dengue Housing (n={total_dengue})', fontweight='bold')
axes[1].pie(housing_chik.values, labels=housing_chik.index, autopct='%1.1f%%',
           colors=plt.cm.Set2(np.linspace(0, 1, len(housing_chik))), startangle=90)
axes[1].set_title(f'Chikungunya Housing (n={total_chik})', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# 6. GEOGRAPHIC LOCATION
# =============================================================================

print("\n" + "="*70)
print("6. GEOGRAPHIC LOCATION ANALYSIS")
print("="*70)

top_places_dengue = df_dengue['Place'].value_counts().head(10)
top_places_chik = df_chik['Place'].value_counts().head(10)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].barh(range(len(top_places_dengue)), top_places_dengue.values, color='#FF6B6B')
axes[0].set_yticks(range(len(top_places_dengue)))
axes[0].set_yticklabels(top_places_dengue.index)
axes[0].set_title(f'Top 10 Dengue Locations (n={total_dengue})', fontweight='bold')
for i, v in enumerate(top_places_dengue.values):
    axes[0].text(v+0.5, i, f'{v} ({v/total_dengue*100:.1f}%)', va='center')

axes[1].barh(range(len(top_places_chik)), top_places_chik.values, color='#4ECDC4')
axes[1].set_yticks(range(len(top_places_chik)))
axes[1].set_yticklabels(top_places_chik.index)
axes[1].set_title(f'Top 10 Chikungunya Locations (n={total_chik})', fontweight='bold')
for i, v in enumerate(top_places_chik.values):
    axes[1].text(v+0.5, i, f'{v} ({v/total_chik*100:.1f}%)', va='center')

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# 7. HOUSEHOLD INCOME
# =============================================================================

print("\n" + "="*70)
print("7. HOUSEHOLD INCOME ANALYSIS")
print("="*70)

income_col = 'Household income/poverty levels: (Tk per year)'
df_dengue['Income_num'] = df_dengue[income_col].astype(str).str.replace(',', '').str.extract(r'(\d+\.?\d*)')[0]
df_dengue['Income_num'] = pd.to_numeric(df_dengue['Income_num'], errors='coerce')
df_dengue['Income_num'] = df_dengue['Income_num'].apply(lambda x: x * 100000 if pd.notna(x) and x < 1000 else x)

df_chik['Income_num'] = df_chik[income_col].astype(str).str.replace(',', '').str.extract(r'(\d+\.?\d*)')[0]
df_chik['Income_num'] = pd.to_numeric(df_chik['Income_num'], errors='coerce')
df_chik['Income_num'] = df_chik['Income_num'].apply(lambda x: x * 100000 if pd.notna(x) and x < 1000 else x)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df_dengue['Income_num'].dropna(), bins=20, color='#FF6B6B', edgecolor='black', alpha=0.7)
axes[0].axvline(df_dengue['Income_num'].mean(), color='red', linestyle='--', label=f'Mean: {df_dengue["Income_num"].mean():,.0f}')
axes[0].set_title(f'Dengue Income Distribution', fontweight='bold')
axes[0].set_xlabel('Income (Tk/year)')
axes[0].legend()

axes[1].hist(df_chik['Income_num'].dropna(), bins=20, color='#4ECDC4', edgecolor='black', alpha=0.7)
axes[1].axvline(df_chik['Income_num'].mean(), color='red', linestyle='--', label=f'Mean: {df_chik["Income_num"].mean():,.0f}')
axes[1].set_title(f'Chikungunya Income Distribution', fontweight='bold')
axes[1].set_xlabel('Income (Tk/year)')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nDengue Income: Mean={df_dengue['Income_num'].mean():,.0f}, Median={df_dengue['Income_num'].median():,.0f} Tk/year")
print(f"Chikungunya Income: Mean={df_chik['Income_num'].mean():,.0f}, Median={df_chik['Income_num'].median():,.0f} Tk/year")

In [ ]:
# =============================================================================
# 8-9. EDUCATION & OVERCROWDING
# =============================================================================

print("\n" + "="*70)
print("8. EDUCATION LEVEL ANALYSIS")
print("="*70)

edu_dengue = df_dengue['Educational level/Literacy'].value_counts()
edu_chik = df_chik['Educational level/Literacy'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].pie(edu_dengue.values, labels=edu_dengue.index, autopct='%1.1f%%',
           colors=plt.cm.Set3(np.linspace(0, 1, len(edu_dengue))), startangle=90)
axes[0].set_title(f'Dengue Education (n={total_dengue})', fontweight='bold')
axes[1].pie(edu_chik.values, labels=edu_chik.index, autopct='%1.1f%%',
           colors=plt.cm.Set3(np.linspace(0, 1, len(edu_chik))), startangle=90)
axes[1].set_title(f'Chikungunya Education (n={total_chik})', fontweight='bold')
plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("9. OVERCROWDING ANALYSIS")
print("="*70)

df_dengue['Overcrowding_num'] = pd.to_numeric(df_dengue['Overcrowding (Person per room)'], errors='coerce')
df_chik['Overcrowding_num'] = pd.to_numeric(df_chik['Overcrowding (Person per room)'], errors='coerce')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df_dengue['Overcrowding_num'].dropna(), bins=10, color='#FF6B6B', edgecolor='black', alpha=0.7)
axes[0].axvline(df_dengue['Overcrowding_num'].mean(), color='red', linestyle='--', label=f'Mean: {df_dengue["Overcrowding_num"].mean():.1f}')
axes[0].set_title(f'Dengue Overcrowding', fontweight='bold')
axes[0].set_xlabel('Persons per Room')
axes[0].legend()

axes[1].hist(df_chik['Overcrowding_num'].dropna(), bins=10, color='#4ECDC4', edgecolor='black', alpha=0.7)
axes[1].axvline(df_chik['Overcrowding_num'].mean(), color='red', linestyle='--', label=f'Mean: {df_chik["Overcrowding_num"].mean():.1f}')
axes[1].set_title(f'Chikungunya Overcrowding', fontweight='bold')
axes[1].set_xlabel('Persons per Room')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Dengue Overcrowding Mean: {df_dengue['Overcrowding_num'].mean():.1f} persons/room")
print(f"Chikungunya Overcrowding Mean: {df_chik['Overcrowding_num'].mean():.1f} persons/room")

In [ ]:
# =============================================================================
# 10-12. WATER/SANITATION, COMMUNITY AWARENESS, PUBLIC HEALTH
# =============================================================================

print("\n" + "="*70)
print("10. WATER STORAGE & SANITATION")
print("="*70)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Sanitation - Dengue
sanitation_dengue = df_dengue['Sanitation facilities'].value_counts()
axes[0, 0].pie(sanitation_dengue.values, labels=sanitation_dengue.index, autopct='%1.1f%%',
              colors=['#4ECDC4', '#FF6B6B'], startangle=90)
axes[0, 0].set_title(f'Dengue Sanitation (n={total_dengue})', fontweight='bold')

# Sanitation - Chikungunya
sanitation_chik = df_chik['Sanitation facilities'].value_counts()
axes[0, 1].pie(sanitation_chik.values, labels=sanitation_chik.index, autopct='%1.1f%%',
              colors=['#4ECDC4', '#FF6B6B'], startangle=90)
axes[0, 1].set_title(f'Chikungunya Sanitation (n={total_chik})', fontweight='bold')

# Water Storage - Dengue
water_dengue = df_dengue['Water storage practices'].value_counts()
axes[1, 0].pie(water_dengue.values, labels=water_dengue.index, autopct='%1.1f%%',
              colors=['#3498DB', '#E74C3C'], startangle=90)
axes[1, 0].set_title(f'Dengue Water Storage (n={total_dengue})', fontweight='bold')

# Water Storage - Chikungunya
water_chik = df_chik['Water storage practices'].value_counts()
axes[1, 1].pie(water_chik.values, labels=water_chik.index, autopct='%1.1f%%',
              colors=['#3498DB', '#E74C3C'], startangle=90)
axes[1, 1].set_title(f'Chikungunya Water Storage (n={total_chik})', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("11. COMMUNITY AWARENESS")
print("="*70)

awareness_col = None
for col in df.columns:
    if 'Community awarness' in col:
        awareness_col = col
        break

if awareness_col:
    awareness_dengue = df_dengue[awareness_col].value_counts()
    awareness_chik = df_chik[awareness_col].value_counts()

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].pie(awareness_dengue.values, labels=awareness_dengue.index, autopct='%1.1f%%',
               colors=['#4ECDC4', '#FF6B6B'], startangle=90)
    axes[0].set_title(f'Dengue Community Awareness (n={total_dengue})', fontweight='bold')
    axes[1].pie(awareness_chik.values, labels=awareness_chik.index, autopct='%1.1f%%',
               colors=['#4ECDC4', '#FF6B6B'], startangle=90)
    axes[1].set_title(f'Chikungunya Community Awareness (n={total_chik})', fontweight='bold')
    plt.tight_layout()
    plt.show()

print("\n" + "="*70)
print("12. PUBLIC HEALTH INTERVENTIONS")
print("="*70)

timing_col = None
coverage_col = None
for col in df.columns:
    if 'Timing of interventions' in col:
        timing_col = col
    if 'Coverage of interventions' in col:
        coverage_col = col

if timing_col:
    timing_dengue = df_dengue[timing_col].value_counts()
    timing_chik = df_chik[timing_col].value_counts()

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].pie(timing_dengue.values, labels=timing_dengue.index, autopct='%1.1f%%',
               colors=['#4ECDC4', '#FF6B6B'], startangle=90)
    axes[0].set_title(f'Dengue Timing of Interventions (n={total_dengue})', fontweight='bold')
    axes[1].pie(timing_chik.values, labels=timing_chik.index, autopct='%1.1f%%',
               colors=['#4ECDC4', '#FF6B6B'], startangle=90)
    axes[1].set_title(f'Chikungunya Timing of Interventions (n={total_chik})', fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# =============================================================================
# 13. TEST ANALYSIS - COMBINED GRAPH
# =============================================================================

print("\n" + "="*70)
print("13. TEST ANALYSIS - COMBINED GRAPH")
print("="*70)

test_columns = ['CBC', 'NSI Antigen', 'Dengue IgG (+) and IgM (+)/(-)',
                'RT PCR test for Chikungunya', 'Chikongunya IgG and IgM']

# Calculate tested percentages
test_data = []
for test in test_columns:
    d_tested = df_dengue[test].notna().sum()
    c_tested = df_chik[test].notna().sum()
    test_data.append({
        'Test': test,
        'Dengue_Tested_Pct': (d_tested/total_dengue)*100,
        'Chik_Tested_Pct': (c_tested/total_chik)*100,
        'Dengue_Tested': d_tested,
        'Chik_Tested': c_tested
    })

test_df = pd.DataFrame(test_data)

fig, ax = plt.subplots(figsize=(14, 7))
x = np.arange(len(test_df))
width = 0.35
bars1 = ax.bar(x - width/2, test_df['Dengue_Tested_Pct'], width, label='Dengue', color='#FF6B6B')
bars2 = ax.bar(x + width/2, test_df['Chik_Tested_Pct'], width, label='Chikungunya', color='#4ECDC4')
ax.set_xticks(x)
ax.set_xticklabels(test_df['Test'], rotation=45, ha='right')
ax.set_ylabel('Percentage Tested (%)')
ax.set_title('Test Coverage Comparison: Dengue vs Chikungunya', fontweight='bold', fontsize=14)
ax.legend()

for bar, pct, cnt in zip(bars1, test_df['Dengue_Tested_Pct'], test_df['Dengue_Tested']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{pct:.1f}%\n(n={cnt})', ha='center', fontsize=8)
for bar, pct, cnt in zip(bars2, test_df['Chik_Tested_Pct'], test_df['Chik_Tested']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{pct:.1f}%\n(n={cnt})', ha='center', fontsize=8)

ax.set_ylim(0, max(test_df['Dengue_Tested_Pct'].max(), test_df['Chik_Tested_Pct'].max()) + 15)
plt.tight_layout()
plt.show()

print("\nTest Coverage Summary:")
print(test_df[['Test', 'Dengue_Tested_Pct', 'Chik_Tested_Pct']].to_string(index=False))

In [ ]:
# =============================================================================
# 14-19. INFRASTRUCTURE & RISK FACTORS
# =============================================================================

print("\n" + "="*70)
print("14-19. INFRASTRUCTURE & RISK FACTORS ANALYSIS")
print("="*70)

# Define analysis items
analysis_items = [
    ('Waste Management', 'Waste management infrastructure (Garbage collection frequency, presence of open dumps)'),
    ('Proximity to Water', 'Proximity to water source (River,lakes, drains)'),
    ('Vector Control', 'Use of vector control measures (bed nets, insecticide sprays, etc)'),
    ('Water Network', 'Water network'),
    ('Long Distance Travel', 'Travel to/from high-incidence areas or endemic regions'),
    ('Waterlogging', 'Waterlogging')
]

fig, axes = plt.subplots(3, 2, figsize=(16, 18))
axes = axes.flatten()

for idx, (title, col_name) in enumerate(analysis_items):
    ax = axes[idx]

    # Convert to Yes/No
    df_dengue_temp = df_dengue[col_name].apply(lambda x: 'Yes' if str(x).strip().lower() == 'yes' else 'No')
    df_chik_temp = df_chik[col_name].apply(lambda x: 'Yes' if str(x).strip().lower() == 'yes' else 'No')

    d_vals = df_dengue_temp.value_counts()
    c_vals = df_chik_temp.value_counts()

    d_yes = d_vals.get('Yes', 0)
    d_no = d_vals.get('No', 0)
    c_yes = c_vals.get('Yes', 0)
    c_no = c_vals.get('No', 0)

    x = np.arange(2)
    width = 0.35
    ax.bar(x - width/2, [d_yes, d_no], width, label='Dengue', color='#FF6B6B')
    ax.bar(x + width/2, [c_yes, c_no], width, label='Chikungunya', color='#4ECDC4')
    ax.set_xticks(x)
    ax.set_xticklabels(['Yes', 'No'])
    ax.set_title(title, fontweight='bold')
    ax.legend()

    # Add percentage labels
    for i, (dv, cv) in enumerate(zip([d_yes, d_no], [c_yes, c_no])):
        ax.text(i - width/2, dv + 1, f'{dv}\n({dv/total_dengue*100:.1f}%)', ha='center', fontsize=8)
        ax.text(i + width/2, cv + 1, f'{cv}\n({cv/total_chik*100:.1f}%)', ha='center', fontsize=8)

plt.tight_layout()
plt.show()

# Print summary
for title, col_name in analysis_items:
    df_dengue_temp = df_dengue[col_name].apply(lambda x: 'Yes' if str(x).strip().lower() == 'yes' else 'No')
    df_chik_temp = df_chik[col_name].apply(lambda x: 'Yes' if str(x).strip().lower() == 'yes' else 'No')
    d_yes = df_dengue_temp.value_counts().get('Yes', 0)
    c_yes = df_chik_temp.value_counts().get('Yes', 0)
    print(f"{title}: Dengue Yes={d_yes} ({d_yes/total_dengue*100:.1f}%), Chikungunya Yes={c_yes} ({c_yes/total_chik*100:.1f}%)")

In [ ]:
# =============================================================================
# 17. OCCUPATION TYPE (Separate detailed view)
# =============================================================================

print("\n" + "="*70)
print("17. OCCUPATION TYPE ANALYSIS")
print("="*70)

occupation_dengue = df_dengue['Occupation Type'].value_counts()
occupation_chik = df_chik['Occupation Type'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].pie(occupation_dengue.values, labels=occupation_dengue.index, autopct='%1.1f%%',
           colors=plt.cm.Set3(np.linspace(0, 1, len(occupation_dengue))), startangle=90)
axes[0].set_title(f'Dengue Occupation (n={total_dengue})', fontweight='bold')
axes[1].pie(occupation_chik.values, labels=occupation_chik.index, autopct='%1.1f%%',
           colors=plt.cm.Set3(np.linspace(0, 1, len(occupation_chik))), startangle=90)
axes[1].set_title(f'Chikungunya Occupation (n={total_chik})', fontweight='bold')
plt.tight_layout()
plt.show()

print("\nDengue Occupations:")
for occ, cnt in occupation_dengue.items():
    print(f"  {occ}: {cnt} ({cnt/total_dengue*100:.1f}%)")
print("\nChikungunya Occupations:")
for occ, cnt in occupation_chik.items():
    print(f"  {occ}: {cnt} ({cnt/total_chik*100:.1f}%)")

In [ ]:
# =============================================================================
# 20 & 24. SEVERITY CLASSIFICATION
# =============================================================================

print("\n" + "="*70)
print("20 & 24. SEVERITY CLASSIFICATION ANALYSIS")
print("="*70)

severity_dengue = df_dengue['Severity Classification'].value_counts()
severity_chik = df_chik['Severity Classification'].value_counts()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Dengue Severity
axes[0].pie(severity_dengue.values, labels=severity_dengue.index, autopct='%1.1f%%',
           colors=plt.cm.RdYlGn(np.linspace(0.1, 0.9, len(severity_dengue))), startangle=90)
axes[0].set_title(f'Dengue Severity (n={total_dengue})', fontweight='bold')

# Chikungunya Severity
axes[1].pie(severity_chik.values, labels=severity_chik.index, autopct='%1.1f%%',
           colors=plt.cm.RdYlGn(np.linspace(0.1, 0.9, len(severity_chik))), startangle=90)
axes[1].set_title(f'Chikungunya Severity (n={total_chik})', fontweight='bold')

# Comparison
all_sev = sorted(set(list(severity_dengue.index) + list(severity_chik.index)))
d_sev_vals = [severity_dengue.get(s, 0) for s in all_sev]
c_sev_vals = [severity_chik.get(s, 0) for s in all_sev]
x = np.arange(len(all_sev))
width = 0.35
axes[2].bar(x - width/2, d_sev_vals, width, label='Dengue', color='#FF6B6B')
axes[2].bar(x + width/2, c_sev_vals, width, label='Chikungunya', color='#4ECDC4')
axes[2].set_xticks(x)
axes[2].set_xticklabels(all_sev, rotation=45, ha='right')
axes[2].legend()
axes[2].set_title('Severity Comparison', fontweight='bold')

plt.tight_layout()
plt.show()

print("\nDengue Severity:")
for sev, cnt in severity_dengue.items():
    print(f"  {sev}: {cnt} ({cnt/total_dengue*100:.1f}%)")
print("\nChikungunya Severity:")
for sev, cnt in severity_chik.items():
    print(f"  {sev}: {cnt} ({cnt/total_chik*100:.1f}%)")

In [ ]:
# =============================================================================
# 22. DRAINAGE SYSTEM
# =============================================================================

print("\n" + "="*70)
print("22. DRAINAGE SYSTEM ANALYSIS")
print("="*70)

drainage_dengue = df_dengue['Drainage system'].value_counts()
drainage_chik = df_chik['Drainage system'].value_counts()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].pie(drainage_dengue.values, labels=drainage_dengue.index, autopct='%1.1f%%',
           colors=['#4ECDC4', '#FF6B6B', '#FFE66D'], startangle=90)
axes[0].set_title(f'Dengue Drainage (n={total_dengue})', fontweight='bold')
axes[1].pie(drainage_chik.values, labels=drainage_chik.index, autopct='%1.1f%%',
           colors=['#4ECDC4', '#FF6B6B', '#FFE66D'], startangle=90)
axes[1].set_title(f'Chikungunya Drainage (n={total_chik})', fontweight='bold')

# Comparison
all_drain = ['Sufficient', 'Insufficient']
d_drain_vals = [drainage_dengue.get(d, 0) for d in all_drain]
c_drain_vals = [drainage_chik.get(d, 0) for d in all_drain]
x = np.arange(len(all_drain))
width = 0.35
axes[2].bar(x - width/2, d_drain_vals, width, label='Dengue', color='#FF6B6B')
axes[2].bar(x + width/2, c_drain_vals, width, label='Chikungunya', color='#4ECDC4')
axes[2].set_xticks(x)
axes[2].set_xticklabels(all_drain)
axes[2].legend()
axes[2].set_title('Drainage Comparison', fontweight='bold')

plt.tight_layout()
plt.show()

suff_dengue = drainage_dengue.get('Sufficient', 0)
insuff_dengue = drainage_dengue.get('Insufficient', 0)
suff_chik = drainage_chik.get('Sufficient', 0)
insuff_chik = drainage_chik.get('Insufficient', 0)

print(f"\nDengue: Sufficient={suff_dengue} ({suff_dengue/total_dengue*100:.1f}%), Insufficient={insuff_dengue} ({insuff_dengue/total_dengue*100:.1f}%)")
print(f"Chikungunya: Sufficient={suff_chik} ({suff_chik/total_chik*100:.1f}%), Insufficient={insuff_chik} ({insuff_chik/total_chik*100:.1f}%)")

In [ ]:
# =============================================================================
# 23. MOSQUITO OUTBREAK
# =============================================================================

print("\n" + "="*70)
print("23. MOSQUITO OUTBREAK ANALYSIS")
print("="*70)

mosquito_dengue = df_dengue['Mosquito outbreak'].value_counts()
mosquito_chik = df_chik['Mosquito outbreak'].value_counts()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].pie(mosquito_dengue.values, labels=mosquito_dengue.index, autopct='%1.1f%%',
           colors=['#FF6B6B', '#4ECDC4'], startangle=90, explode=[0.05]*len(mosquito_dengue))
axes[0].set_title(f'Dengue Mosquito Outbreak (n={total_dengue})', fontweight='bold')
axes[1].pie(mosquito_chik.values, labels=mosquito_chik.index, autopct='%1.1f%%',
           colors=['#FF6B6B', '#4ECDC4'], startangle=90, explode=[0.05]*len(mosquito_chik))
axes[1].set_title(f'Chikungunya Mosquito Outbreak (n={total_chik})', fontweight='bold')

all_mos = ['Yes', 'No']
d_mos = [mosquito_dengue.get(m, 0) for m in all_mos]
c_mos = [mosquito_chik.get(m, 0) for m in all_mos]
x = np.arange(len(all_mos))
width = 0.35
axes[2].bar(x - width/2, d_mos, width, label='Dengue', color='#FF6B6B')
axes[2].bar(x + width/2, c_mos, width, label='Chikungunya', color='#4ECDC4')
axes[2].set_xticks(x)
axes[2].set_xticklabels(all_mos)
axes[2].legend()
axes[2].set_title('Mosquito Outbreak Comparison', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\nDengue Mosquito Outbreak: Yes={mosquito_dengue.get('Yes', 0)} ({(mosquito_dengue.get('Yes', 0)/total_dengue*100):.1f}%)")
print(f"Chikungunya Mosquito Outbreak: Yes={mosquito_chik.get('Yes', 0)} ({(mosquito_chik.get('Yes', 0)/total_chik*100):.1f}%)")

In [ ]:
# =============================================================================
# 25. AGE GROUP ANALYSIS
# =============================================================================

print("\n" + "="*70)
print("25. AGE GROUP ANALYSIS")
print("="*70)

df_dengue['Age_numeric'] = df_dengue['Age'].astype(str).str.extract(r'(\d+)').astype(float)
df_chik['Age_numeric'] = df_chik['Age'].astype(str).str.extract(r'(\d+)').astype(float)

age_bins = [0, 15, 30, 45, 60, 75, 90]
age_labels = ['0-15', '16-30', '31-45', '46-60', '61-75', '76-90']
df_dengue['Age_Group'] = pd.cut(df_dengue['Age_numeric'], bins=age_bins, labels=age_labels)
df_chik['Age_Group'] = pd.cut(df_chik['Age_numeric'], bins=age_bins, labels=age_labels)

age_dengue = df_dengue['Age_Group'].value_counts().sort_index()
age_chik = df_chik['Age_Group'].value_counts().sort_index()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

colors_age = plt.cm.RdYlGn(np.linspace(0.1, 0.9, len(age_dengue)))
axes[0].pie(age_dengue.values, labels=age_dengue.index, autopct='%1.1f%%',
           colors=colors_age, startangle=90)
axes[0].set_title(f'Dengue Age Groups (n={total_dengue})', fontweight='bold')

axes[1].pie(age_chik.values, labels=age_chik.index, autopct='%1.1f%%',
           colors=colors_age, startangle=90)
axes[1].set_title(f'Chikungunya Age Groups (n={total_chik})', fontweight='bold')

# Comparison
x = np.arange(len(age_labels))
width = 0.35
d_age_vals = [age_dengue.get(a, 0) for a in age_labels]
c_age_vals = [age_chik.get(a, 0) for a in age_labels]
axes[2].bar(x - width/2, d_age_vals, width, label='Dengue', color='#FF6B6B')
axes[2].bar(x + width/2, c_age_vals, width, label='Chikungunya', color='#4ECDC4')
axes[2].set_xticks(x)
axes[2].set_xticklabels(age_labels, rotation=45)
axes[2].legend()
axes[2].set_title('Age Group Comparison', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\nDengue Mean Age: {df_dengue['Age_numeric'].mean():.1f} years")
print(f"Chikungunya Mean Age: {df_chik['Age_numeric'].mean():.1f} years")
print("\nAge Group Distribution:")
for age in age_labels:
    d_cnt = age_dengue.get(age, 0)
    c_cnt = age_chik.get(age, 0)
    print(f"  {age}: Dengue={d_cnt} ({d_cnt/total_dengue*100:.1f}%), Chikungunya={c_cnt} ({c_cnt/total_chik*100:.1f}%)")

In [ ]:
# =============================================================================
# COMPREHENSIVE SUMMARY
# =============================================================================

print("\n" + "="*80)
print("COMPREHENSIVE ANALYSIS SUMMARY")
print("="*80)

print(f"""
TOTAL PATIENTS: {total_all}
  Dengue: {total_dengue} ({total_dengue/total_all*100:.1f}%)
  Chikungunya: {total_chik} ({total_chik/total_all*100:.1f}%)

1. GENDER: D-M:{male_dengue} F:{female_dengue} | C-M:{male_chik} F:{female_chik}

2. TOP SYMPTOMS:
   Dengue: {symptom_df.loc[symptom_df['Dengue_Pct'].idxmax(), 'Symptom']} ({symptom_df['Dengue_Pct'].max():.1f}%)
   Chik: {symptom_df.loc[symptom_df['Chik_Pct'].idxmax(), 'Symptom']} ({symptom_df['Chik_Pct'].max():.1f}%)

3. TEMPERATURE: D-Hot:{hot_dengue} Cold:{cold_dengue} | C-Hot:{hot_chik} Cold:{cold_chik}

5. HOUSING: D-{housing_dengue.index[0]} | C-{housing_chik.index[0]}

6. TOP LOCATION: D-{top_places_dengue.index[0]} | C-{top_places_chik.index[0]}

7. INCOME: D-Mean:{df_dengue['Income_num'].mean():,.0f} | C-Mean:{df_chik['Income_num'].mean():,.0f}

8. EDUCATION: D-{edu_dengue.index[0]} | C-{edu_chik.index[0]}

9. OVERCROWDING: D-Mean:{df_dengue['Overcrowding_num'].mean():.1f} | C-Mean:{df_chik['Overcrowding_num'].mean():.1f}

10. SANITATION (Adequate): D:{sanitation_dengue.get('Adequate', 0)} ({(sanitation_dengue.get('Adequate', 0)/total_dengue*100):.1f}%) | C:{sanitation_chik.get('Adequate', 0)} ({(sanitation_chik.get('Adequate', 0)/total_chik*100):.1f}%)

15. OCCUPATION: D-{occupation_dengue.index[0]} | C-{occupation_chik.index[0]}

20. SEVERITY: D-{severity_dengue.index[0]} | C-{severity_chik.index[0]}

22. DRAINAGE (Sufficient): D:{suff_dengue} ({suff_dengue/total_dengue*100:.1f}%) | C:{suff_chik} ({suff_chik/total_chik*100:.1f}%)

23. MOSQUITO OUTBREAK: D-Yes:{mosquito_dengue.get('Yes', 0)} ({(mosquito_dengue.get('Yes', 0)/total_dengue*100):.1f}%) | C-Yes:{mosquito_chik.get('Yes', 0)} ({(mosquito_chik.get('Yes', 0)/total_chik*100):.1f}%)

25. AGE: D-Mean:{df_dengue['Age_numeric'].mean():.1f} | C-Mean:{df_chik['Age_numeric'].mean():.1f}
""")

print("="*80)
print("ANALYSIS COMPLETE!")
print("="*80)